# 8. Transformers

Convolutional neural network approaches have long dominated image processing. In this notebook, we introduce the 
**Transformer architecture**, which has revolutionised both natural language processing and computer vision, and 
is currently the dominant paradigm across much of deep learning.

In this notebook, we will get an idea of

* the attention mechanism and how it works,
* multi-head attention,
* positional encoding,
* the Vision Transformer (ViT) for image classification,
* and compare transformers with CNNs on MNIST.

[The Illustrated Transformer](https://jalammar.github.io/illustrated-transformer/) is a great introduction to the 
topic and provides many nice illustrations, some of which we borrow in this notebook.

Keywords: ```Transformers```, ```Attention```, ```Self-Attention```, ```Multi-Head Attention```, 
```Vision Transformer (ViT)```, ```Patch Embedding```, ```Positional Encoding```

***

## Attention is All You Need

**Transformers** were introduced in the seminal [Attention is All You Need](https://proceedings.neurips.cc/paper_files/paper/2017/file/3f5ee243547dee91fbd053c1c4a845aa-Paper.pdf) paper by Vaswani et al. in 2017 and have become the dominant architecture in deep learning. 

What makes transformers particularly powerful? For convolutional neural networks (CNNs), we have seen that they
- process images with convolutional filters over **local receptive fields**, i.e. e.g. $3 \times 3$ windows of neighbouring pixels and
- learn long-range dependencies through many convolutional and pooling layers.

The underlying design choice or **inductive bias** for CNNs is hence the idea of grouping information of 
neigbourhing cells / pixels together, allowing for **locality** and **translation equivariance**.

Transformers introduce a more general design choice, overcoming some of the limitations of CNNs and other neural network architectures:
- **Self-attention**: Every position can **attend** to / be processed with every other position (think of "position" as "pixel" for the time being)
- **Global receptive field** from the first layer: long-range dependencies are immediately learnable
- **Learned relationships** rather than hard-coded inductive biases like the local receptive field
- **Parallelizable** computation (unlike recurrent neural networks, for example)


## Understanding the Attention Mechanism

To illustrate the idea of self-attention, consider an example where we want to translate the sentence:

```The animal didn't cross the street because it was too tired.```

In a first step, we separate the sentence into **tokens**, a (pre-defined) basic unit of text / characters that the transformer is supposed to process. The illustrations below shows the different tokens in different lines. Self-attentions learns what other words (tokens) are important to process e.g. the word (token) ```it```, with darker shades of orange  higher importance / weight like for ```animal```.

<center><img src="images/transformer_one_head_example.png" alt="Attention example" width="250"/> <br> Image source: <a href="https://jalammar.github.io/images/t/transformer_self-attention_visualization.png">The Illustrated Transformer</a> <br> </center>


The **attention mechanism** which allows the model to focus on relevant parts of the input when making predictions. Think of attention like a **database query**:
- **Query (Q)**: What am I looking for? In our example, that's ```it```.
- **Key (K)**: What does each token / position represent? In our example, all other words (tokens), think "semantic content". 
- **Value (V)**: What information does each token / position contain? In our example, think of the "value" of information for the query.

The attention mechanism computes which values to retrieve based on the similarity between queries and keys.

Let's implement and understand transformers step by step!

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import math
import tensorflow as tf
from tensorflow import keras

seed = 123

### Step 1: Scaled Dot-Product Attention

The fundamental attention operation is the **scaled dot-product attention**

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

with
- $Q \in \mathbb{R}^{n \times d_k}$ are the **queries** (what we're looking for)
- $K \in \mathbb{R}^{m \times d_k}$ are the **keys** (what each position represents)
- $V \in \mathbb{R}^{m \times d_v}$ are the **values** (actual content at each position)
- $d_k$ is the dimension of queries/keys
- $n$ is the number of query positions
- $m$ is the number of key/value positions
- and the softmax operation we encountered previously $\text{softmax}(z)_i = \frac{\exp(z_i)}{\sum_{j=1}^{C} \exp (z_j)}$


#### How this works

1. **Compute similarity**: $QK^T$ gives similarity scores between queries and keys
2. **Scaling**: Divide by $\sqrt{d_k}$ to prevent too small/big values, particularly **vanishing gradients**
3. **Normalise**: Applying the softmax provides attention weights between 0 and 1
4. **Aggregate**: Weighted sum of values using attention weights

### Step 2: Self-Attention

In **self-attention**, $Q$, $K$, and $V$ all come from the same input, like our sentence example above:
- Each token / position attends to all positions (including itself)
- Learns relationships between different parts of the input

We will implement attention as a custom Keras layer by subclassing ```keras.layers.Layer``` to see this in action. 
As seen many times in this course, there's also a convenient built-in ```keras.layers.MultiHeadAttention``` layer, too.

In [ ]:
class ScaledDotProductAttention(keras.layers.Layer):
    def __init__(self, temperature, **kwargs):
        super().__init__(**kwargs)
        self.temperature = temperature  # sqrt(d_k)

    def call(self, q, k, v, mask=None):
        """
        Args:
            q: Queries (batch_size, n_heads, len_q, d_k)
            k: Keys (batch_size, n_heads, len_k, d_k)
            v: Values (batch_size, n_heads, len_v, d_v)
            mask: Optional mask
        Returns:
            output: Attention output (batch_size, n_heads, len_q, d_v)
            attn:   Attention weights (batch_size, n_heads, len_q, len_k)
        """
        # Compute attention scores: Q * K^T / sqrt(d_k)
        attn = tf.matmul(q, k, transpose_b=True) / self.temperature

        # Apply mask if provided (for padding or causality)
        if mask is not None:
            attn = tf.where(mask == 0, tf.constant(-1e9, dtype=attn.dtype), attn)

        # Apply softmax to get attention weights
        attn = tf.nn.softmax(attn, axis=-1)

        # Apply attention weights to values
        output = tf.matmul(attn, v)

        return output, attn

### Step 3: Multi-Head Attention

How about using several attention layers? **Multi-head attention** runs multiple attention operations in parallel:

$$\text{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, ..., \text{head}_h)W^O$$

where each head is

$$\text{head}_i = \text{Attention}(QW_i^Q, KW_i^K, VW_i^V)$$

In the illustration below, the two heads are shown with attention values in orange and green for processing ```it```. 
The first head *attends* to / puts large weights on ```The animal```, while the second focuses more on ```tired```.

<center><img src="images/transformer_two_head_example.png" alt="Multi-head attention example" width="300"/> <br> Image source: <a href="https://jalammar.github.io/images/t/transformer_self-attention_visualization_2.png">The Illustrated Transformer</a> <br> </center>

Similar to using several convolutional filters, multiple heads offer the possibility that
- different heads can attend to different patterns,
- one head might focus on local patterns, another on global structure, or
- model capacity and expressiveness are increased.

In [ ]:
class MultiHeadAttention(keras.layers.Layer):
    def __init__(self, n_heads, d_model, dropout=0.1, **kwargs):
        super().__init__(**kwargs)

        assert d_model % n_heads == 0, "d_model must be divisible by n_heads"

        self.n_heads = n_heads
        self.d_model = d_model
        self.d_k = d_model // n_heads  # Dimension per head

        # Linear projections for Q, K, V
        self.w_q = keras.layers.Dense(d_model)
        self.w_k = keras.layers.Dense(d_model)
        self.w_v = keras.layers.Dense(d_model)

        # Output projection
        self.w_o = keras.layers.Dense(d_model)

        # Attention mechanism
        self.attention = ScaledDotProductAttention(temperature=self.d_k ** 0.5)

        self.dropout = keras.layers.Dropout(dropout)
        self.layer_norm = keras.layers.LayerNormalization()

    def call(self, q, k, v, mask=None, training=None):
        batch_size = tf.shape(q)[0]
        len_q = tf.shape(q)[1]
        len_k = tf.shape(k)[1]
        len_v = tf.shape(v)[1]

        # Save residual for later
        residual = q

        # Linear projections and split into heads
        # (batch, len, d_model) -> (batch, len, n_heads, d_k) -> (batch, n_heads, len, d_k)
        q = tf.reshape(self.w_q(q), [batch_size, len_q, self.n_heads, self.d_k])
        q = tf.transpose(q, [0, 2, 1, 3])
        k = tf.reshape(self.w_k(k), [batch_size, len_k, self.n_heads, self.d_k])
        k = tf.transpose(k, [0, 2, 1, 3])
        v = tf.reshape(self.w_v(v), [batch_size, len_v, self.n_heads, self.d_k])
        v = tf.transpose(v, [0, 2, 1, 3])

        # Apply attention
        output, attn = self.attention(q, k, v, mask=mask)

        # Concatenate heads
        # (batch, n_heads, len_q, d_k) -> (batch, len_q, n_heads, d_k) -> (batch, len_q, d_model)
        output = tf.transpose(output, [0, 2, 1, 3])
        output = tf.reshape(output, [batch_size, len_q, self.d_model])

        # Output projection
        output = self.dropout(self.w_o(output), training=training)

        # Add residual and normalize
        output = self.layer_norm(output + residual)

        return output, attn

### Step 4: Positional Encoding

So far, our attention layers have no notion of order or position! While in CNNs the position is implicit,
because neighboring pixels are related, all positions are treated equally in attention. To address this,
**positional encodings** are used to provide transformers with information about position, i.e. how far
two words are apart from each other.

The original transformer uses sinusoidal positional encoding

$$PE_{(pos, 2i)} = \sin\left(\frac{pos}{10000^{2i/d}}\right)$$
$$PE_{(pos, 2i+1)} = \cos\left(\frac{pos}{10000^{2i/d}}\right)$$

where $pos$ is the position, $i$ is the dimension and different frequencies are used for different dimensions.

In [ ]:
class PositionalEncoding(keras.layers.Layer):
    def __init__(self, d_model, max_len=5000, **kwargs):
        super().__init__(**kwargs)

        # Create positional encoding matrix
        pe = np.zeros((max_len, d_model), dtype=np.float32)
        position = np.arange(0, max_len, dtype=np.float32)[:, np.newaxis]
        div_term = np.exp(
            np.arange(0, d_model, 2, dtype=np.float32)
            * (-math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = np.sin(position * div_term)
        pe[:, 1::2] = np.cos(position * div_term)

        # Add batch dimension and store as a non-trainable constant
        self.pe = tf.constant(pe[np.newaxis, ...])

    def call(self, x):
        # x: (batch_size, seq_len, d_model)
        seq_len = tf.shape(x)[1]
        return x + self.pe[:, :seq_len, :]

In [ ]:
pe_layer = PositionalEncoding(d_model=100, max_len=50)
encoding = pe_layer.pe[0].numpy()

plt.imshow(encoding.T, aspect='auto', cmap='RdBu')

plt.xlabel('Position')
plt.ylabel('Dimension')
plt.title('Sinusoidal Positional Encoding')

plt.colorbar()

plt.show()

#### Note 
that the first dimensions have faster oscillations and thus capture short-range / detailed information, 
and the later dimensions have fewer oscillations and capture long-range / broader information.

***

## The Vision Transformer

So far, our discussion has been very much focused on NLP examples. The [An Image Is Worth 16x16 Words: 
Transformers For Image Recognition At Scale](https://arxiv.org/pdf/2010.11929/100) paper by Dosovitskiy et al. from 2021 
showed how to apply transformers to image classification! The **Vision Transformer (ViT)** treats an image as a sequence of patches,
as illustrated below.

<center><img src="images/transformer_ViT_illustration.png" alt="Vision Transformer illustration" width="650"/> <br> Image source: <a href="https://arxiv.org/pdf/2010.11929/100">Vision Transformer paper</a> <br> </center>

In the following, we will adapt the ViT for our MNIST classification example. The architecture will 
look something like this:

```
Image (28x28)
↓
Split into patches (7x7 patches → 16 patches of 7x7)
↓
Flatten patches (16 patches of 49 pixels)
↓
Linear projection (16 patches → 16 embeddings)
↓
Add [CLS] token + positional embeddings
↓
Transformer Encoder (L layers)
↓
Extract [CLS] token
↓
Classification head (MLP)
↓
Output (10 classes for MNIST)
```

The key components here will be

1. **Patch Embedding**: Divide an image into patches and linearly project them
2. **[CLS] Token**: Special learnable token for classification
3. **Positional Embedding**: Learnable position information
4. **Transformer Encoder**: Stack of multi-head attention + FFNN layers
5. **Classification Head**: MLP on [CLS] token output

In [ ]:
class PatchEmbedding(keras.layers.Layer):
    """Split image into patches and embed them"""

    def __init__(self, img_size=28, patch_size=7, embed_dim=64, **kwargs):
        super().__init__(**kwargs)
        self.img_size = img_size
        self.patch_size = patch_size
        self.embed_dim = embed_dim
        self.n_patches = (img_size // patch_size) ** 2  # Number of patches

        # Conv2D with kernel=patch_size and stride=patch_size
        # extracts non-overlapping patches and projects them to embed_dim
        self.proj = keras.layers.Conv2D(
            filters=embed_dim,
            kernel_size=patch_size,
            strides=patch_size,
        )

    def call(self, x):
        # x: (batch_size, height, width, channels)  -- channel-last (Keras default)
        x = self.proj(x)  # (batch_size, n_patches_h, n_patches_w, embed_dim)

        # Flatten the spatial dims into a sequence of patches
        batch_size = tf.shape(x)[0]
        x = tf.reshape(x, [batch_size, self.n_patches, self.embed_dim])
        return x  # (batch_size, n_patches, embed_dim)


class TransformerEncoderBlock(keras.layers.Layer):
    def __init__(self, d_model, n_heads, mlp_ratio=4.0, dropout=0.1, **kwargs):
        super().__init__(**kwargs)

        # Multi-head self-attention
        self.attn = MultiHeadAttention(n_heads, d_model, dropout)

        # Feed-forward network
        mlp_hidden_dim = int(d_model * mlp_ratio)
        self.mlp = keras.Sequential([
            keras.layers.Dense(mlp_hidden_dim, activation='gelu'),  # GELU is smoother than ReLU
            keras.layers.Dropout(dropout),
            keras.layers.Dense(d_model),
            keras.layers.Dropout(dropout),
        ])

        self.norm1 = keras.layers.LayerNormalization()
        self.norm2 = keras.layers.LayerNormalization()

    def call(self, x, training=None):
        # Self-attention with residual connection (pre-norm)
        x_norm = self.norm1(x)
        attn_out, attn_weights = self.attn(x_norm, x_norm, x_norm, training=training)
        x = x + attn_out

        # Feed-forward with residual connection
        x = x + self.mlp(self.norm2(x), training=training)

        return x, attn_weights

class VisionTransformer(keras.Model):
    def __init__(
        self,
        img_size=28,
        patch_size=7,
        num_classes=10,
        embed_dim=64,
        depth=4,
        n_heads=4,
        mlp_ratio=4.0,
        dropout=0.1,
        **kwargs,
    ):
        super().__init__(**kwargs)

        # Patch embedding
        self.patch_embed = PatchEmbedding(img_size, patch_size, embed_dim)
        n_patches = self.patch_embed.n_patches

        # [CLS] token - learnable classification token
        self.cls_token = self.add_weight(
            name='cls_token',
            shape=(1, 1, embed_dim),
            initializer=keras.initializers.TruncatedNormal(stddev=0.02),
            trainable=True,
        )

        # Positional embeddings (learnable)
        self.pos_embed = self.add_weight(
            name='pos_embed',
            shape=(1, n_patches + 1, embed_dim),
            initializer=keras.initializers.TruncatedNormal(stddev=0.02),
            trainable=True,
        )
        self.pos_drop = keras.layers.Dropout(dropout)

        # Transformer encoder blocks
        self.blocks = [
            TransformerEncoderBlock(embed_dim, n_heads, mlp_ratio, dropout, name=f'block_{i}')
            for i in range(depth)
        ]

        self.norm = keras.layers.LayerNormalization()

        # Classification head (softmax probabilities, matching the rest of the course)
        self.head = keras.layers.Dense(num_classes, activation='softmax')

    def call(self, x, training=None, return_attention=False):
        batch_size = tf.shape(x)[0]

        # Patch embedding
        x = self.patch_embed(x)  # (batch_size, n_patches, embed_dim)

        # Prepend [CLS] token
        cls_tokens = tf.tile(self.cls_token, [batch_size, 1, 1])
        x = tf.concat([cls_tokens, x], axis=1)  # (batch_size, n_patches+1, embed_dim)

        # Add positional encoding
        x = x + self.pos_embed
        x = self.pos_drop(x, training=training)

        # Apply transformer blocks
        attention_maps = []
        for block in self.blocks:
            x, attn = block(x, training=training)
            if return_attention:
                attention_maps.append(attn)

        x = self.norm(x)

        # Extract [CLS] token for classification
        cls_output = x[:, 0]

        # Classification probabilities
        probs = self.head(cls_output)

        if return_attention:
            return probs, attention_maps
        return probs

Let's load the MNIST data as we've done before

In [ ]:
mnist_data = np.load('data/mnist_data_5k.npy', allow_pickle=True).astype('float32')
mnist_targets = np.load('data/mnist_labels_5k.npy', allow_pickle=True).astype('int64')

mnist_data = mnist_data.reshape(-1, 28, 28, 1) / 255.0

num_train = 3000
num_val = 1000
num_test = 1000

x_train = mnist_data[:num_train]
y_train = mnist_targets[:num_train]

x_val = mnist_data[num_train:num_train + num_val]
y_val = mnist_targets[num_train:num_train + num_val]

x_test = mnist_data[-num_test:]
y_test = mnist_targets[-num_test:]

print(f"Training set:   {x_train.shape}")
print(f"Validation set: {x_val.shape}")
print(f"Test set:       {x_test.shape}")

And train our ViT model on MNIST

In [ ]:
batch_size = 64
num_epochs = 60
learning_rate = 0.001

img_size = 28
patch_size = 7  # 28/7 = 4x4 = 16 patches
embed_dim = 64
depth = 10      # Number of transformer blocks
n_heads = 8     # Number of attention heads
num_classes = 10

keras.utils.set_random_seed(seed)

# Create the model
vit = VisionTransformer(
    img_size=img_size,
    patch_size=patch_size,
    num_classes=num_classes,
    embed_dim=embed_dim,
    depth=depth,
    n_heads=n_heads,
    name='ViT',
)

# Build the model so we can call .summary() before training
vit.build(input_shape=(None, img_size, img_size, 1))

# Cosine-decay learning rate schedule (matches the spirit of CosineAnnealingLR)
steps_per_epoch = num_train // batch_size
schedule = keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=learning_rate,
    decay_steps=num_epochs * steps_per_epoch,
    alpha=1e-6 / learning_rate,  # eta_min / initial_lr
)

# AdamW with weight decay (typical choice for transformers)
optimizer = keras.optimizers.AdamW(
    learning_rate=schedule,
    weight_decay=0.01,
)

vit.compile(
    optimizer=optimizer,
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

vit_history = vit.fit(
    x=x_train, y=y_train,
    batch_size=batch_size,
    validation_data=(x_val, y_val),
    epochs=num_epochs,
)

test_loss, test_acc = vit.evaluate(x_test, y_test, verbose=0)
print(f"\nTest accuracy: {test_acc:.4f}")

vit.summary()

### Training Visualisation

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 4))

axs[0].plot(vit_history.history['accuracy'], label='Training')
axs[0].plot(vit_history.history['val_accuracy'], label='Validation')

axs[0].set_xlabel('Epoch')
axs[0].set_ylabel('Accuracy')
axs[0].set_title('Vision Transformer - Accuracy')
axs[0].legend()
axs[0].grid(True, alpha=0.3)

axs[1].plot(vit_history.history['loss'], label='Training')
axs[1].plot(vit_history.history['val_loss'], label='Validation')

axs[1].set_xlabel('Epoch')
axs[1].set_ylabel('Loss')
axs[1].set_title('Vision Transformer - Loss')
axs[1].legend()
axs[1].grid(True, alpha=0.3)

plt.show()

### Visualize Attention Maps

One of the most interesting aspects of transformers is that we can **visualise what the model is attending to**!

Because our ```VisionTransformer``` accepts a ```return_attention=True``` flag, we can call it directly outside of ```fit``` to retrieve the per-head attention weights.

In [ ]:
def visualise_attention(model, image, target, n_heads=4):
    """Visualise attention maps for a single image."""
    # Add a batch dimension; image has shape (28, 28, 1)
    image_tensor = image[np.newaxis, ...]
    probs, attention_maps = model(image_tensor, training=False, return_attention=True)
    pred = int(tf.argmax(probs, axis=1).numpy()[0])

    # Get attention from the last transformer block
    # Shape: (batch=1, n_heads, n_tokens, n_tokens)
    last_attn = attention_maps[-1][0].numpy()  # (n_heads, n_tokens, n_tokens)

    # Attention from the [CLS] token (index 0) to each patch token.
    # This shows which patches matter for the classification decision.
    cls_attn = last_attn[:, 0, 1:]  # (n_heads, n_patches)

    # Reshape to image grid (4x4 for patch_size=7 on 28x28)
    n_patches_per_side = int(cls_attn.shape[1] ** 0.5)

    # fig, axes = plt.subplots(3, n_heads // 3 + 1, figsize=(6, 6))
    fig, axes = plt.subplots(1, n_heads + 1, figsize=(12, 8))
    axes = axes.flatten()

    # Show original image
    axes[0].imshow(image.squeeze())
    axes[0].set_title(f"Target: {target}, Pred: {pred}")
    axes[0].axis('off')

    # Show attention for each head
    for i in range(n_heads):
        attn_map = cls_attn[i].reshape(n_patches_per_side, n_patches_per_side)
        im = axes[i + 1].imshow(attn_map, cmap='hot', vmin=0, vmax=1)
        axes[i + 1].set_title(f'Head {i + 1}')
        axes[i + 1].axis('off')

    fig.colorbar(im, ax=axes[i + 1], fraction=0.046, pad=0.04)


    plt.tight_layout()
    plt.show()

In [ ]:
for i in range(5):
    visualise_attention(vit, x_test[i], y_test[i], n_heads=8)

### Compare with a CNN

Let's compare our Vision Transformer with a CNN, similar to the one we have seen before.

In [ ]:
keras.utils.set_random_seed(seed)

cnn = keras.Sequential([
    keras.layers.InputLayer(input_shape=(28, 28, 1)),
    keras.layers.Conv2D(filters=32, kernel_size=(3, 3), padding='same', activation='relu'),
    keras.layers.MaxPooling2D(pool_size=(2, 2)),
    keras.layers.Conv2D(filters=64, kernel_size=(3, 3), padding='same', activation='relu'),
    keras.layers.MaxPooling2D(pool_size=(2, 2)),
    keras.layers.Flatten(),
    keras.layers.Dense(units=128, activation='relu'),
    keras.layers.Dropout(0.5),
    keras.layers.Dense(units=10, activation='softmax'),
], name='CNN')

cnn.compile(
    optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

cnn_history = cnn.fit(
    x=x_train, y=y_train,
    batch_size=batch_size,
    validation_data=(x_val, y_val),
    epochs=num_epochs,
)

cnn_test_loss, cnn_test_acc = cnn.evaluate(x_test, y_test, verbose=0)

### Comparison

In [ ]:
fig, axs = plt.subplots(3, 1, figsize=(6, 10))

# Accuracy comparison
axs[0].plot(cnn_history.history['accuracy'], label='CNN Train', c='blue', linestyle='-', alpha=0.7)
axs[0].plot(cnn_history.history['val_accuracy'], label='CNN Val', c='blue', linestyle='--', alpha=0.7)
axs[0].plot(vit_history.history['accuracy'], label='ViT Train', c='red', linestyle='-', alpha=0.7)
axs[0].plot(vit_history.history['val_accuracy'], label='ViT Val', c='red', linestyle='--', alpha=0.7)
# axs[0].set_xlabel('Epoch')
axs[0].set_ylabel('Accuracy')
axs[0].set_title('Training Accuracy: CNN vs ViT')
axs[0].legend()
axs[0].grid(True, alpha=0.3)

# Loss comparison
axs[1].plot(cnn_history.history['loss'], label='CNN Train', c='blue', linestyle='-', alpha=0.7)
axs[1].plot(cnn_history.history['val_loss'], label='CNN Val', c='blue', linestyle='--', alpha=0.7)
axs[1].plot(vit_history.history['loss'], label='ViT Train', c='red', linestyle='-', alpha=0.7)
axs[1].plot(vit_history.history['val_loss'], label='ViT Val', c='red', linestyle='--', alpha=0.7)
# axs[1].set_xlabel('Epoch')
axs[1].set_ylabel('Loss')
axs[1].set_title('Training Loss: CNN vs ViT')
axs[1].legend()
axs[1].grid(True, alpha=0.3)

# Model comparison table
axs[2].axis('off')
comparison_data = [
    ['Metric', 'CNN', 'ViT'],
    ['Test Accuracy', f"{cnn_test_acc:.4f}", f"{test_acc:.4f}"],
    ['Parameters', f"{cnn.count_params():,}", f"{vit.count_params():,}"],
]

table = axs[2].table(cellText=comparison_data, loc='center', cellLoc='left')
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 2)

# Style header row
for i in range(3):
    table[(0, i)].set_facecolor('#40466e')
    table[(0, i)].set_text_props(weight='bold', color='white')

# axs[2].set_title('Model Comparison', fontsize=12, weight='bold')

plt.show()

***
## Summary

| Aspect | CNN | Transformer (ViT) |
|--------|-----|------------------|
| Receptive Field | Local → Global | Global from start |
| Inductive Bias | Strong (locality) | Weak (learned) |
| Data Efficiency | High | Low (needs more data) |
| Interpretability | Limited | High (attention maps) |
| Scalability | Good | Excellent |
| Parameters | Fewer | More |

So, when could it be a good idea to use transformers?

**Good for:**
- Large datasets 
- Transfer learning scenarios
- When interpretability matters
- Long-range dependencies
- Multi-modal tasks

But **not ideal for:**
- Very small datasets
- Limited computational resources
- When strong locality bias helps